In [3]:

# ============================================================
# VRU v23 -- EXPLICIT REGISTER ARCHITECTURE
# Dylan Michael Scott -- Horizon Tech
#
# Evolved from v21/v22. The structural ceiling in v14-v22 was
# a task formulation failure, not an architecture failure.
# The unstructured hidden state could not reliably preserve
# digit values and carry info simultaneously -- they interfere
# in a shared vector.
#
# v23 fixes this by construction:
#   - 6 dedicated register slots in hidden state
#   - PHI-field (memory, oscillatory) drives digit registers
#   - ALPHA-field (carry, contraction) drives carry register
#   - Fields interact via additive phase injection ONLY
#   - PHI x ALPHA = 1.0 -- geometric dual identity
#   - Zero gates (no sigmoid/forget/input gates)
#   - ~4x parameter efficiency vs LSTM
#
# Register layout in h [hidden_dim total, hidden_dim % 6 == 0]:
#   [0   : R]    ones_A  -- units digit of operand A  (PHI-field)
#   [R   : 2R]   tens_A  -- tens  digit of operand A  (PHI-field)
#   [2R  : 3R]   ones_B  -- units digit of operand B  (PHI-field)
#   [3R  : 4R]   tens_B  -- tens  digit of operand B  (PHI-field)
#   [4R  : 5R]   carry   -- carry field               (ALPHA-field)
#   [5R  : 6R]   answer  -- running answer accumulation (PHI-field)
#   where R = hidden_dim // 6
#
# Improvements from v21/v22:
#   + Explicit register partitioning (solves structural ceiling)
#   + Separate ALPHA-field for carry (no digit/carry interference)
#   + Additive phase injection between fields (replaces agreement gate)
#   + Spectral clipping on W_h (RHO_H) and W_c (RHO_C)
#   + Four-tier state buffer manager with circuit breaker
#   + Register norm cap (prevents hidden state explosion)
#   + Three-criterion mastery gate (answer + holdout + carry)
#   + Between-stage state consolidation (4 phases)
#   + Regression detection on all prior stages
#   + JSONL audit trail (immutable, append-only)
#   + Git-compatible checkpoint protocol
#
# Curriculum (6 stages):
#   1: single-digit no carry  (0-9   + 0-9)
#   2: single-digit carry     (5-9   + 5-9)
#   3: two-digit no carry     (10-49 + 10-49)
#   4: two-digit carry        (50-99 + 50-99)
#   5: mixed two-digit        (10-99 + 10-99)
#   6: three-digit chains     (100-999 + 100-999)
#
# Input format (proven from v21):
#   73+18=ones:11,carry:1,tens:9,ans:91
#
# Single Colab cell. Saves to Google Drive. Auto-resumes.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import os
import json
import time
from datetime import datetime, timezone

# ── PATHS ─────────────────────────────────────────────────────────────────────
DRIVE_DIR  = '/content/drive/MyDrive/dppu_vru'
CKPT_DIR   = os.path.join(DRIVE_DIR, 'v23_checkpoints')
AUDIT_PATH = os.path.join(DRIVE_DIR, 'v23_audit.jsonl')
LOG_PATH   = os.path.join(DRIVE_DIR, 'v23_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,  exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"VRU v23 | Device: {device}")

# ── GEOMETRIC CONSTANTS ───────────────────────────────────────────────────────
# Core IP of Horizon Tech.
# True values used internally. Never log, never checkpoint.
_PHI   = 4.0 / math.pi    # 1.2732395447351626...
_ALPHA = math.pi / 4.0    # 0.7853981633974483...
assert abs(_PHI * _ALPHA - 1.0) < 1e-12, "Dual identity violated"
_RHO_H = 1.0 / _PHI       # W_h spectral clip target
_RHO_C = _PHI              # W_c spectral clip target

# ── CONFIG ────────────────────────────────────────────────────────────────────
CFG = dict(
    hidden           = 192,     # Must be divisible by 6
    lr               = 3e-4,
    batch            = 32,
    max_steps        = 50_000,
    grad_clip        = 1.0,
    weight_decay     = 1e-5,
    tf_start         = 0.95,
    tf_min           = 0.10,
    tf_decay         = 0.9998,
    log_every        = 100,
    eval_every       = 500,
    probe_every      = 1000,
    ckpt_every       = 2000,
    start_stage      = 1,
    mastery_answer   = 0.85,
    mastery_holdout  = 0.80,
    mastery_carry    = 0.75,
    mastery_windows  = 3,
    regression_warn  = 0.80,
    regression_crit  = 0.70,
    consol_min_epochs  = 5,
    consol_min_windows = 3,
    consol_drift_mult  = 1.5,
    consol_prune_thr   = 1e-4,
    buf_util_thr     = 0.85,
    buf_grad_warn    = 3.0,
    buf_grad_emerg   = 10.0,
    buf_cb_thr       = 3,
)
REG_DIM = CFG['hidden'] // 6   # 32 per register slot

# ── AUDIT + LOG ───────────────────────────────────────────────────────────────
def audit(event, **kw):
    rec = {'event': event, 'ts': datetime.now(timezone.utc).isoformat(), **kw}
    with open(AUDIT_PATH, 'a') as f:
        f.write(json.dumps(rec) + '\n')

def log(msg):
    line = f"[{datetime.now().strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')

# ── TOKENIZER ────────────────────────────────────────────────────────────────
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']
    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        for c in '0123456789+-=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv    = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
    @property
    def vocab_size(self): return len(self.vocab)
    def encode(self, text, bos=True, eos=True):
        ids = [self.vocab.get(c, self.vocab['<unk>']) for c in text]
        if bos: ids = [self.bos_id] + ids
        if eos: ids = ids + [self.eos_id]
        return ids
    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

TOK = MathTokenizer()

# ── CURRICULUM ────────────────────────────────────────────────────────────────
STAGES = {
    1: (0,   9,   0,   9,   'single-digit no carry'),
    2: (5,   9,   5,   9,   'single-digit carry'),
    3: (10,  49,  10,  49,  'two-digit no carry'),
    4: (50,  99,  50,  99,  'two-digit carry'),
    5: (10,  99,  10,  99,  'mixed two-digit'),
    6: (100, 999, 100, 999, 'three-digit chains'),
}
CARRY_STAGES = {2, 4, 5, 6}

def make_problem(stage):
    lo_a, hi_a, lo_b, hi_b, _ = STAGES[stage]
    a = random.randint(lo_a, hi_a)
    b = random.randint(lo_b, hi_b)
    ones  = (a % 10) + (b % 10)
    carry = ones // 10
    tens  = (a // 10) + (b // 10) + carry
    ans   = a + b
    text  = f"{a}+{b}=ones:{ones},carry:{carry},tens:{tens},ans:{ans}"
    return text, a, b, ans, carry

def make_holdout_problem(stage):
    """Holdout: avoid training-distribution clustering."""
    text, a, b, ans, carry = make_problem(stage)
    tries = 0
    while (a % 3 == 0 and b % 3 == 0) and tries < 20:
        text, a, b, ans, carry = make_problem(stage)
        tries += 1
    return text, a, b, ans, carry

def collate_batch(problems):
    max_len = max(len(p[0]) for p in problems)
    padded  = [p[0] + [TOK.pad_id] * (max_len - len(p[0])) for p in problems]
    return torch.tensor(padded, dtype=torch.long)

# ── VRU CELL ──────────────────────────────────────────────────────────────────
class VRUCell(nn.Module):
    """
    Explicit register cell with dual-field geometry.

    Hidden state h is partitioned into 6 register slots (R = reg_dim each):
        ones_A, tens_A, ones_B, tens_B  ->  PHI-field (W_h recurrence)
        carry                           ->  ALPHA-field (W_c recurrence, separate)
        answer                          ->  PHI-field (aggregates all)

    PHI-field and ALPHA-field interact ONLY via additive phase injection.
    No gates. Spectral clipping applied after each optimizer step.

    Update order (read-parallel, write-serial):
        Phase 1: parallel reads of all register slices
        Phase 2: compute new_carry from ALPHA-field
        Phase 3: compute new_digits from PHI-field + carry injection
        Phase 4: serial writes (carry first, then digits, then answer)
    """

    def __init__(self, vocab_size, hidden, reg_dim):
        super().__init__()
        self.hidden  = hidden
        self.reg_dim = reg_dim

        # Input
        self.embed = nn.Embedding(vocab_size, hidden)
        self.W_in  = nn.Linear(hidden, hidden, bias=True)

        # PHI-field: drives all registers except carry
        # Operates on (hidden - reg_dim) = first 5 register slots
        self.W_h = nn.Linear(hidden - reg_dim, hidden - reg_dim, bias=False)

        # ALPHA-field: drives carry register only (separate recurrent field)
        self.W_c = nn.Linear(reg_dim, reg_dim, bias=False)

        # Output
        self.W_out = nn.Linear(hidden, vocab_size, bias=True)

        self._init()

    def _init(self):
        nn.init.orthogonal_(self.W_h.weight)
        nn.init.orthogonal_(self.W_c.weight)
        with torch.no_grad():
            self.W_h.weight.mul_(_RHO_H)
            self.W_c.weight.mul_(_RHO_C * 0.5)  # Conservative carry init
        nn.init.xavier_uniform_(self.W_in.weight)
        nn.init.zeros_(self.W_in.bias)
        nn.init.xavier_uniform_(self.W_out.weight)
        nn.init.zeros_(self.W_out.bias)

    def clip_spectral(self):
        """Clip W_h to RHO_H, W_c to RHO_C. Call after optimizer step."""
        with torch.no_grad():
            for W, rho in [(self.W_h.weight, _RHO_H), (self.W_c.weight, _RHO_C)]:
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                W.copy_(U @ torch.diag(S.clamp(max=rho)) @ Vh)

    def step(self, x_tok, h):
        """
        x_tok : (B,)
        h     : (B, hidden)
        returns logits (B, V), h_new (B, hidden)
        """
        R = self.reg_dim

        # ── PHASE 1: PARALLEL READS ───────────────────────────────────────
        x_emb  = self.embed(x_tok)                  # (B, hidden)
        x_proj = self.W_in(x_emb)                   # (B, hidden)

        h_digs = h[:, :self.hidden - R]             # first 5 slots
        h_carr = h[:, 4*R:5*R]                      # carry slot only

        # ── PHASE 2: ALPHA-FIELD -- new carry ────────────────────────────
        c_rec     = self.W_c(h_carr)                 # (B, R)
        x_c       = x_proj[:, 4*R:5*R]
        # Additive phase injection from tens_B (positional anchor for carry)
        # tens_B is slice [3R:4R] of h_digs
        tens_b    = h_digs[:, 3*R:4*R]
        new_carry = torch.tanh(
            _ALPHA * (c_rec + x_c) + 0.1 * tens_b  # injection, not gating
        )

        # ── PHASE 3: PHI-FIELD -- new digits ─────────────────────────────
        h_rec   = self.W_h(h_digs)                  # (B, hidden-R)
        x_d     = x_proj[:, :self.hidden - R]
        new_digs = torch.tanh(_PHI * h_rec + x_d)   # (B, hidden-R)

        # ── PHASE 4: SERIAL WRITES (carry first) ─────────────────────────
        h_new = torch.cat([new_digs, new_carry], dim=-1)  # (B, hidden)

        logits = self.W_out(h_new)                   # (B, V)
        return logits, h_new

    def init_h(self, B):
        return torch.zeros(B, self.hidden, device=device)


# ── REGISTER NORM CAP ─────────────────────────────────────────────────────────
def cap_registers(h, R):
    """
    ALPHA-contraction if any register slot exceeds geometric ceiling.
    Ceiling = 1 / ALPHA. Preserves field ratios.
    """
    CEIL = 1.0 / _ALPHA
    with torch.no_grad():
        for i in range(6):
            sl    = h[:, i*R:(i+1)*R]
            norms = sl.norm(dim=-1, keepdim=True)
            over  = norms > CEIL
            if over.any():
                scale = torch.where(over, _ALPHA * CEIL / norms.clamp(1e-8),
                                    torch.ones_like(norms))
                h[:, i*R:(i+1)*R] = sl * scale
    return h


# ── FOUR-TIER STATE BUFFER MANAGER ────────────────────────────────────────────
class StateManager:
    """
    Four-tier memory pressure management with circuit breaker.
    Tier 1: register decay        (always, zero cost)
    Tier 2: EMA compression       (util > 85%)
    Tier 3: ALPHA contraction     (grad > 3x EMA)
    Tier 4: emergency reset       (grad > 10x EMA)
    CB:     reduce LR             (3 consecutive Tier-3 fires)
    """
    def __init__(self):
        self.grad_ema = 1.0
        self.t3_count = 0

    def step(self, h, grad_norm, R):
        self.grad_ema = 0.95 * self.grad_ema + 0.05 * grad_norm
        ratio = grad_norm / max(self.grad_ema, 1e-8)

        h = self._t1_decay(h, R)

        if self._util(h, R) > CFG['buf_util_thr']:
            h = self._t2_compress(h)

        if ratio > CFG['buf_grad_warn']:
            self.t3_count += 1
            if self.t3_count >= CFG['buf_cb_thr']:
                self.t3_count = 0
                return h, 'CB'
            with torch.no_grad():
                h = h * _ALPHA
        else:
            self.t3_count = 0

        if ratio > CFG['buf_grad_emerg']:
            with torch.no_grad():
                carry = h[:, 4*R:5*R].clone()
                h.zero_()
                h[:, 4*R:5*R] = carry
            return h, 'EMERG'

        return h, 'OK'

    def _t1_decay(self, h, R):
        with torch.no_grad():
            carry = h[:, 4*R:5*R]
            stale = carry.norm(dim=-1, keepdim=True) < 1e-3
            if stale.any():
                h[:, 4*R:5*R] = torch.where(stale, carry * _ALPHA, carry)
        return h

    def _t2_compress(self, h):
        with torch.no_grad():
            h = h * (_PHI / (1.0 + _PHI))
        return h

    def _util(self, h, R):
        return h.norm(dim=-1).mean().item() / (1.0 / _ALPHA)


# ── MASTERY GATE ──────────────────────────────────────────────────────────────
class MasteryGate:
    """Three-criterion gate. All must hold for N consecutive eval windows."""
    def __init__(self):
        self.hist = []   # list of (ans_acc, hold_acc, carry_acc)

    def record(self, ans, hold, carry):
        self.hist.append((ans, hold, carry))

    def check(self, stage, prior_accs):
        N = CFG['mastery_windows']
        if len(self.hist) < N:
            return False, 'not_enough_windows'
        recent = self.hist[-N:]

        ans_ok   = all(w[0] >= CFG['mastery_answer']  for w in recent)
        hold_ok  = recent[-1][1] >= CFG['mastery_holdout']
        carry_ok = True
        if stage in CARRY_STAGES:
            carry_ok = all(w[2] >= CFG['mastery_carry'] for w in recent)

        for i, acc in enumerate(prior_accs):
            if acc < CFG['regression_crit']:
                return False, f'regression_critical_stage_{i+1}'
            if acc < CFG['regression_warn']:
                return False, f'regression_warning_stage_{i+1}'

        ok = ans_ok and hold_ok and carry_ok
        return ok, {'ans': ans_ok, 'hold': hold_ok, 'carry': carry_ok}

    def reset(self):
        self.hist = []


# ── STATE CONSOLIDATION ───────────────────────────────────────────────────────
class Consolidator:
    """
    Between-stage state consolidation.
    Fires: after mastery confirmed, before stage advance.
    Phases: snapshot -> identify drift -> stabilize -> prune.
    """
    def __init__(self):
        self.last_epoch   = 0
        self.last_windows = 0
        self.norm_hist    = [[] for _ in range(6)]
        self.grad_emas    = [1.0] * 6
        self.init_vals    = [None] * 6
        self.running      = False

    def ready(self, epoch, windows):
        e_ok  = (epoch   - self.last_epoch)   >= CFG['consol_min_epochs']
        w_ok  = (windows - self.last_windows) >= CFG['consol_min_windows']
        return e_ok and w_ok and not self.running

    def update_stats(self, model):
        R = model.cell.reg_dim
        for i in range(6):
            w = model.cell.W_h.weight if i < 5 else model.cell.W_c.weight
            ri = i if i < 5 else 0
            norm = w[ri*R:(ri+1)*R, :].norm().item() if i < 5 else w.norm().item()
            self.norm_hist[i].append(norm)
            if len(self.norm_hist[i]) > 20:
                self.norm_hist[i].pop(0)

    def run(self, model, epoch, windows):
        self.running = True
        log("  [CONSOLIDATION] Starting...")

        # Phase 1: Snapshot
        R = model.cell.reg_dim
        norms = []
        for i in range(5):
            norms.append(model.cell.W_h.weight[i*R:(i+1)*R,:].norm().item())
        norms.append(model.cell.W_c.weight.norm().item())
        log(f"  [P1] Norms: {[f'{n:.3f}' for n in norms]}")

        # Phase 2: Identify drift
        drifted = []
        for i in range(6):
            hist = self.norm_hist[i]
            if len(hist) >= 10:
                base   = sum(hist[:5])  / 5
                recent = sum(hist[-5:]) / 5
                if base > 0 and recent > base * CFG['consol_drift_mult']:
                    drifted.append(i)
        log(f"  [P2] Drifted: {drifted}")

        # Phase 3: Stabilize drifted
        stabilized = []
        with torch.no_grad():
            for i in drifted:
                if i < 5:
                    W = model.cell.W_h.weight
                    sl = W[i*R:(i+1)*R, :]
                else:
                    sl = model.cell.W_c.weight

                if self.init_vals[i] is None:
                    self.init_vals[i] = sl.clone()
                # PHI weights recent; ALPHA anchors prior stable
                new_val = _PHI * sl + _ALPHA * self.init_vals[i]
                sl.copy_(new_val)
                self.init_vals[i] = new_val.clone()
                stabilized.append(i)
        log(f"  [P3] Stabilized: {stabilized}")

        # Phase 4: Prune dead registers
        pruned = []
        for i in range(6):
            if self.grad_emas[i] < CFG['consol_prune_thr']:
                with torch.no_grad():
                    if i < 5:
                        model.cell.W_h.weight[i*R:(i+1)*R,:].zero_()
                    else:
                        model.cell.W_c.weight.zero_()
                pruned.append(i)
        log(f"  [P4] Pruned: {pruned}")

        self.last_epoch   = epoch
        self.last_windows = windows
        self.running      = False
        audit('consolidation', epoch=epoch, drifted=drifted,
              stabilized=stabilized, pruned=pruned)
        log("  [CONSOLIDATION] Complete.")


# ── VRU MODEL ─────────────────────────────────────────────────────────────────
class VRUModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.cell = VRUCell(TOK.vocab_size, CFG['hidden'], REG_DIM)

    def forward(self, ids, tf_ratio=1.0):
        B, T = ids.shape
        h = self.cell.init_h(B)
        logits_all = []
        h_all      = []
        for t in range(T - 1):
            if t == 0 or random.random() < tf_ratio:
                x = ids[:, t]
            else:
                x = logits_all[-1].argmax(-1)
            lg, h = self.cell.step(x, h)
            h     = cap_registers(h, REG_DIM)
            logits_all.append(lg)
            h_all.append(h)
        return torch.stack(logits_all, 1), torch.stack(h_all, 1)

    def generate(self, prompt_ids, max_new=60):
        self.eval()
        with torch.no_grad():
            ids = prompt_ids.to(device)
            h   = self.cell.init_h(1)
            out = list(ids[0].tolist())
            for t in range(ids.size(1) - 1):
                _, h = self.cell.step(ids[0, t:t+1], h)
            x = ids[0, -1:]
            for _ in range(max_new):
                lg, h = self.cell.step(x, h)
                nx = lg.argmax(-1)
                out.append(nx.item())
                if nx.item() == TOK.eos_id:
                    break
                x = nx
        return out


# ── ANSWER-ONLY LOSS ──────────────────────────────────────────────────────────
def _ans_prefix():
    """Token ids for 'ans:' -- cached once."""
    return TOK.encode('ans:', bos=False, eos=False)

ANS_PREFIX = _ans_prefix()

def build_answer_mask(ids):
    """
    Boolean mask: True only at answer token positions (after 'ans:').
    v14 bug was computing loss on input echoes. This is the fix.
    """
    B, T = ids.shape
    mask = torch.zeros(B, T - 1, dtype=torch.bool)
    plen = len(ANS_PREFIX)
    for b in range(B):
        seq = ids[b].tolist()
        for i in range(len(seq) - plen):
            if seq[i:i+plen] == ANS_PREFIX:
                ans_start = i + plen
                for j in range(ans_start, min(T - 1, len(seq))):
                    if j + 1 < T and seq[j+1] != TOK.pad_id:
                        mask[b, j] = True
                break
    return mask

def compute_loss(logits, ids):
    """Cross entropy on answer tokens only."""
    targets = ids[:, 1:]
    mask    = build_answer_mask(ids).to(device)
    if mask.sum() == 0:
        return F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1), ignore_index=TOK.pad_id
        ), 0
    return F.cross_entropy(logits[mask], targets[mask]), mask.sum().item()


# ── EVALUATION ────────────────────────────────────────────────────────────────
def _extract(text, key):
    if key not in text:
        return None
    try:
        part = text.split(key)[-1].split(',')[0].split('<')[0].strip()
        digits = ''.join(c for c in part if c.isdigit())
        return int(digits) if digits else None
    except:
        return None

def evaluate(model, stage, n=200, holdout=False):
    model.eval()
    ok_ans = ok_carry = total = carry_total = 0
    with torch.no_grad():
        for _ in range(n):
            fn = make_holdout_problem if holdout else make_problem
            text, a, b, true_ans, true_carry = fn(stage)
            prompt = TOK.encode(f"{a}+{b}=", eos=False)
            p_ids  = torch.tensor([prompt], dtype=torch.long, device=device)
            gen    = TOK.decode(model.generate(p_ids))
            if _extract(gen, 'ans:') == true_ans:
                ok_ans += 1
            if stage in CARRY_STAGES:
                carry_total += 1
                if _extract(gen, 'carry:') == true_carry:
                    ok_carry += 1
            total += 1
    model.train()
    ans_acc   = ok_ans / total
    carry_acc = ok_carry / carry_total if carry_total else 1.0
    return ans_acc, carry_acc


# ── REGISTER PROBES ───────────────────────────────────────────────────────────
REG_NAMES = ['ones_A', 'tens_A', 'ones_B', 'tens_B', 'carry ', 'answer']

def probe(model, stage, n=5):
    model.eval()
    log("  ── PROBES ──────────────────────────────────────────")
    with torch.no_grad():
        for _ in range(n):
            _, a, b, true_ans, _ = make_problem(stage)
            p_ids = torch.tensor(
                [TOK.encode(f"{a}+{b}=", eos=False)], dtype=torch.long, device=device)
            h = model.cell.init_h(1)
            for t in range(p_ids.size(1) - 1):
                _, h = model.cell.step(p_ids[0, t:t+1], h)
            R = REG_DIM
            norms = [h[0, i*R:(i+1)*R].norm().item() for i in range(6)]
            nstr  = ' | '.join(f"{REG_NAMES[i]}={norms[i]:.3f}" for i in range(6))
            gen   = TOK.decode(model.generate(p_ids))
            pred  = _extract(gen, 'ans:')
            mark  = '✓' if pred == true_ans else '✗'
            log(f"  {mark} {a}+{b}={true_ans}  pred={pred}  [{nstr}]")
    log("  ────────────────────────────────────────────────────")
    model.train()


# ── CHECKPOINTING ─────────────────────────────────────────────────────────────
def save_ckpt(model, opt, state, tag):
    path = os.path.join(CKPT_DIR, f'v23_{tag}.pt')
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'state': state}, path)
    log(f"  [CKPT] {path}")
    audit('checkpoint', tag=tag, step=state['step'])

def load_latest(model, opt):
    if not os.path.isdir(CKPT_DIR):
        return None
    files = sorted(
        [f for f in os.listdir(CKPT_DIR) if f.startswith('v23_') and f.endswith('.pt')],
        key=lambda f: os.path.getmtime(os.path.join(CKPT_DIR, f)), reverse=True
    )
    if not files:
        return None
    path = os.path.join(CKPT_DIR, files[0])
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model'])
    opt.load_state_dict(ckpt['opt'])
    log(f"  [RESUME] {path}")
    return ckpt['state']


# ── TRAINING ──────────────────────────────────────────────────────────────────
def train():
    log("=" * 64)
    log("VRU v23  --  Explicit Register Architecture")
    log(f"Hidden: {CFG['hidden']}  |  Reg dim: {REG_DIM}  |  Device: {device}")
    log(f"PHI x ALPHA = {_PHI * _ALPHA:.15f}  (dual identity)")
    log(f"Vocab: {TOK.vocab_size}  |  Params: estimating...")
    log("=" * 64)

    model = VRUModel().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    log(f"Parameters: {n_params:,}")

    opt = torch.optim.AdamW(
        model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=2000, T_mult=2, eta_min=1e-5)

    state_mgr   = StateManager()
    gate        = MasteryGate()
    consolidator= Consolidator()

    state = dict(step=0, epoch=0, stage=CFG['start_stage'],
                 mwin=0, tf=CFG['tf_start'], prior_accs=[],
                 consol_count=0, mastery_hist=[])

    loaded = load_latest(model, opt)
    if loaded:
        state.update(loaded)
        gate.hist = state.get('mastery_hist', [])
        log(f"  Resumed: step={state['step']}, stage={state['stage']}")

    stage_name = STAGES[state['stage']][4]
    log(f"Stage {state['stage']}: {stage_name}\n")
    audit('start', step=state['step'], stage=state['stage'])

    model.train()
    run_loss = 0.0

    while state['step'] < CFG['max_steps']:

        # ── Batch ──────────────────────────────────────────────────────────
        problems = [make_problem(state['stage']) for _ in range(CFG['batch'])]
        raw      = [(TOK.encode(t), a, b, ans, c) for t,a,b,ans,c in problems]
        ids      = collate_batch(raw).to(device)

        # ── Forward ────────────────────────────────────────────────────────
        opt.zero_grad()
        logits, h_seq = model(ids, tf_ratio=state['tf'])
        loss, n_ans   = compute_loss(logits, ids)

        if n_ans == 0:
            state['step'] += 1
            continue

        # ── Backward ───────────────────────────────────────────────────────
        loss.backward()
        g_norm = nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']).item()
        model.cell.clip_spectral()
        opt.step()
        sched.step()

        # ── State buffer ───────────────────────────────────────────────────
        h_last = h_seq[:, -1, :].detach()
        h_last, buf_st = state_mgr.step(h_last, g_norm, REG_DIM)
        if buf_st == 'CB':
            old_lr = opt.param_groups[0]['lr']
            for pg in opt.param_groups: pg['lr'] *= 0.5
            log(f"  [CB] LR {old_lr:.2e} -> {opt.param_groups[0]['lr']:.2e}")
            audit('circuit_breaker', step=state['step'], old_lr=old_lr)

        # ── Teacher forcing decay ──────────────────────────────────────────
        state['tf'] = max(CFG['tf_min'], state['tf'] * CFG['tf_decay'])

        run_loss    += loss.item()
        state['step']+= 1
        state['epoch'] = state['step'] // 100

        # ── Log ────────────────────────────────────────────────────────────
        if state['step'] % CFG['log_every'] == 0:
            avg = run_loss / CFG['log_every']
            run_loss = 0.0
            lr = opt.param_groups[0]['lr']
            log(f"S{state['step']:5d} | Stage {state['stage']} | "
                f"Loss {avg:.4f} | GN {g_norm:.3f} | LR {lr:.2e} | TF {state['tf']:.3f}")
            audit('step', step=state['step'], loss=avg, gnorm=g_norm,
                  stage=state['stage'], lr=lr)

        # ── Eval window ────────────────────────────────────────────────────
        if state['step'] % CFG['eval_every'] == 0:
            log(f"\n  ── EVAL step {state['step']} ──────────────────────────")

            ans_acc,  carry_acc  = evaluate(model, state['stage'], n=200)
            hold_acc, hold_carry = evaluate(model, state['stage'], n=200, holdout=True)

            log(f"  Train  ans={ans_acc:.3f}  carry={carry_acc:.3f}")
            log(f"  Holdout ans={hold_acc:.3f}  carry={hold_carry:.3f}")

            gate.record(ans_acc, hold_acc, carry_acc)
            state['mwin'] += 1
            consolidator.update_stats(model)

            audit('eval', step=state['step'], stage=state['stage'],
                  ans=ans_acc, carry=carry_acc, hold=hold_acc)

            # ── Mastery gate check ────────────────────────────────────────
            passed, info = gate.check(state['stage'], state['prior_accs'])

            if passed:
                log(f"  ✓ MASTERY GATE PASSED -- Stage {state['stage']}!")
                log(f"    {info}")

                # Consolidation before advance
                if consolidator.ready(state['epoch'], state['mwin']):
                    consolidator.run(model, state['epoch'], state['mwin'])
                    state['consol_count'] += 1

                state['prior_accs'].append(ans_acc)
                old_stage = state['stage']
                state['stage'] += 1
                gate.reset()

                if state['stage'] > 6:
                    log("\n  ✓ ALL 6 STAGES COMPLETE!")
                    save_ckpt(model, opt, state, 'final')
                    audit('complete', step=state['step'])
                    break

                log(f"  → Stage {state['stage']}: {STAGES[state['stage']][4]}")
                audit('advance', step=state['step'],
                      from_stage=old_stage, to_stage=state['stage'])
                save_ckpt(model, opt, state, f'after_stage{old_stage}')

            else:
                log(f"  ✗ Gate: {info}")

        # ── Probes ─────────────────────────────────────────────────────────
        if state['step'] % CFG['probe_every'] == 0:
            probe(model, state['stage'])

        # ── Periodic checkpoint ────────────────────────────────────────────
        if state['step'] % CFG['ckpt_every'] == 0:
            state['mastery_hist'] = gate.hist
            save_ckpt(model, opt, state, f'step{state["step"]}')

    log(f"\nDone. Final stage: {state['stage']} | Steps: {state['step']}")


# ── RUN ───────────────────────────────────────────────────────────────────────
train()

Mounted at /content/drive
VRU v23 | Device: cuda
[08:34:54] ================================================================
[08:34:54] VRU v23  --  Explicit Register Architecture
[08:34:54] Hidden: 192  |  Reg dim: 32  |  Device: cuda
[08:34:54] PHI x ALPHA = 1.000000000000000  (dual identity)
[08:34:54] Vocab: 47  |  Params: estimating...
[08:34:54] ================================================================
[08:34:54] Parameters: 81,775
[08:34:59] Stage 1: single-digit no carry



RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation: [torch.cuda.FloatTensor [32, 192]], which is output 0 of CatBackward0, is at version 6; expected version 0 instead. Hint: enable anomaly detection to find the operation that failed to compute its gradient, with torch.autograd.set_detect_anomaly(True).

In [2]:
from google.colab import auth
auth.authenticate_user()